# Example 4: Preflight 体检 + 诊断信息查看

展示两个新功能：
1. **Preflight**：求解前用 `syntaxcheck` / `datacheck` 拦截坏 INP
2. **Diagnostics**：失败时从 `JobOutcome.diagnostics` 读取真实错误原因

In [7]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec,generate_from_inp_files, degenerate_from_array

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/abaqus.bat'
CWD = os.getcwd()

## 使用 preflight 体检现有 INP

设置 `preflight="syntaxcheck"`，Abaqus 只检查关键字语法，不占求解 token。通过后才进入正式求解；失败直接标记 `PREFLIGHT_FAILED`，省下整次求解的时间。

In [ ]:
# 构造一个带 preflight 的 base spec
base_spec = JobSpec(
	job_name = "preflight_example",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "existing_inp",
		source_path = "./examples/cae_file/planar_stress_main.inp",
		options = {"staging_mode": "copy", "resolve_includes": True}
	),
	# preflight = "syntaxcheck",  # <-- 求解前检查 INP 
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises"},
				{"result_name": "max_displacement"},
			]
		)
	]
)

specs = generate_from_inp_files(
	["./examples/cae_file/scenarios/planar_stress_scenario_1.inp",
	"./examples/cae_file/scenarios/planar_stress_scenario_2.inp"],
	base_spec, naming="stem"
)

In [ ]:
processor = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = os.path.join(CWD, "examples/04_PreflightAndDiagnostics/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

outcomes = processor.run_batch(num_parallel_jobs=2)

Output()

## 查看诊断信息

无论作业成功还是失败，`JobOutcome` 现在都携带 `diagnostics` 字段：
- `diagnostics["sta_verdict"]` — .sta 文件的权威终态标记
- `diagnostics["errors"]` — .msg/.dat 中收割的真实错误行（去重+截断，最多 5 条）
- `diagnostics["error_total"]` / `["warning_total"]` — 错误/警告总数
- `diagnostics["increments"]` — 完成的增量步数
- `diagnostics["solver_type"]` — `standard` | `explicit`

`JobOutcome.error` 也已升级：失败时直接显示第一条收割到的 ERROR 行。

In [12]:
# 检查每个作业的结果和诊断
from ABQflow import outcomes_to_dict

results = outcomes_to_dict(outcomes)
for job_name, data in results.items():
	status = data["status"]
	if status == "COMPLETED":
		print(f"OK {job_name}: COMPLETED")
		print(f"   max_stress_mises = {data.get('max_stress_mises', 'N/A')}")
	elif status == "PREFLIGHT_FAILED":
		print(f"XX {job_name}: PREFLIGHT_FAILED")
		print(f"   error: {data.get('error', 'N/A')}")
	else:
		print(f"XX {job_name}: {status}")
		print(f"   error: {data.get('error', 'N/A')}")
	# 查看诊断详情（IMP-02）
	diag = data.get("diagnostics")
	if diag:
		print(f"   .sta verdict: {diag.get('sta_verdict')}")
		print(f"   solver: {diag.get('solver_type')}, increments: {diag.get('increments')}")
		if diag.get("errors"):
			for e in diag["errors"]:
				print(f"   harvested ERROR: {e[:150]}")

XX planar_stress_scenario_1: SIMULATION_FAILED
   error: N/A
   .sta verdict: INDETERMINATE
   solver: unknown, increments: 0
   harvested ERROR: ***ERROR: INVALID FLOAT VALUE
OK planar_stress_scenario_2: COMPLETED
   max_stress_mises = 6787.890625


## 批量体检模式（preflight_only）

如果你有 100 个来路不明的 INP，先全量体检一遍，修好再正式求解。
`preflight_only=True`：只跑 preparation + preflight，跳过求解和提取。

In [ ]:
inspector = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = os.path.join(CWD, "examples/PreflightAndDiagnostics/output_inspect"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
	preflight_only = True,  # <-- 只检查，不求解
)

inspector.prepare()
inspect_results = inspector.run_batch(num_parallel_jobs=4)

passed = sum(1 for o in inspect_results if o.status == "COMPLETED")
failed = sum(1 for o in inspect_results if o.status == "PREFLIGHT_FAILED")
print(f"Preflight results: {passed} passed, {failed} failed")
for o in inspect_results:
	if o.status == "PREFLIGHT_FAILED":
		print(f"  XX {o.job_name}: {o.error}")

Output()

Preflight results: 1 passed, 1 failed
  XX planar_stress_scenario_1: None
